# 面试题：时间冲突怎样驱动记忆更新？

记忆更新要比较事件时间、来源与置信度，不能按到达顺序覆盖。权威来源优先于模型推测，较新事件时间优先于旧事件，撤销优先于普通更新；保留历史以便审计。

## 真实案例

用户上午在上海、下午迁至北京，但离线客户端晚到上海事件；六条记录还包括权威删除和模型推测。

## 基线

基线按网络到达顺序覆写地址。

## 结果解读

resolver 输出生效记录和冲突历史。

## 失败案例

晚到旧事件不能回滚较新的权威北京地址。

In [1]:
events = [{'id':'T1','value':'上海','time':9,'source':'authority','deleted':False}, {'id':'T2','value':'北京','time':15,'source':'authority','deleted':False}, {'id':'T3','value':'上海','time':9,'source':'authority','deleted':False}, {'id':'T4','value':'广州','time':16,'source':'model','deleted':False}, {'id':'T5','value':'北京','time':17,'source':'authority','deleted':True}, {'id':'T6','value':'深圳','time':18,'source':'authority','deleted':False}]  # 构造六条含事件时间、来源和撤销语义的更新。
print('时间事件:', events)  # 输出乱序到达的地址更新。
print('教学说明：time 代表可信服务端事件时间，客户端时间在生产中需额外防伪。')  # 说明时间来源。

时间事件: [{'id': 'T1', 'value': '上海', 'time': 9, 'source': 'authority', 'deleted': False}, {'id': 'T2', 'value': '北京', 'time': 15, 'source': 'authority', 'deleted': False}, {'id': 'T3', 'value': '上海', 'time': 9, 'source': 'authority', 'deleted': False}, {'id': 'T4', 'value': '广州', 'time': 16, 'source': 'model', 'deleted': False}, {'id': 'T5', 'value': '北京', 'time': 17, 'source': 'authority', 'deleted': True}, {'id': 'T6', 'value': '深圳', 'time': 18, 'source': 'authority', 'deleted': False}]
教学说明：time 代表可信服务端事件时间，客户端时间在生产中需额外防伪。


In [2]:
arrival_value = None  # 初始化按到达顺序的错误基线。
for event in events:  # 按网络接收顺序处理事件。
    arrival_value = None if event['deleted'] else event['value']  # 让晚到/低可信事件直接覆盖当前值。
print('到达顺序基线地址:', arrival_value)  # 输出最后一条恰好正确但不可审计的视图。
print('基线问题：若 T3 晚到在末尾，会把北京错误回滚上海。')  # 说明到达顺序不可靠。

到达顺序基线地址: 深圳
基线问题：若 T3 晚到在末尾，会把北京错误回滚上海。


In [3]:
rank = {'authority':2,'model':1}  # 定义权威事件高于模型推测的来源优先级。
def resolve(items):  # 定义按事件时间、来源和删除语义解决冲突的函数。
    ordered = sorted(items, key=lambda item:(item['time'], rank[item['source']]))  # 用事件时间和来源排序而非网络到达顺序。
    current = None  # 初始化当前生效记录。
    history = []  # 初始化可审计冲突历史。
    for item in ordered:  # 依次处理时间有序的事件。
        if item['source'] == 'model' and current is not None and current['source'] == 'authority':  # 阻止模型推测覆盖已有权威事实。
            history.append((item['id'], current['value']))  # 记录模型冲突被拒绝后的生效值。
            continue  # 继续处理下一条事件。
        current = None if item['deleted'] else item  # 删除事件清空当前值，普通事件成为新版本。
        history.append((item['id'], None if current is None else current['value']))  # 记录每个事件后的生效值。
    return current, history  # 返回最终生效记录和变化轨迹。

In [4]:
current, history = resolve(events)  # 对乱序事件运行时间感知 resolver。
print('id | 处理后生效地址')  # 输出冲突解决轨迹表标题。
for item in history:  # 遍历每条事件处理后的当前视图。
    print(item[0], item[1])  # 输出事件与生效值中间量。
print('最终权威地址:', current['value'])  # 输出最终按时间/来源解析的地址。

id | 处理后生效地址
T1 上海
T3 上海
T2 北京
T4 北京
T5 None
T6 深圳
最终权威地址: 深圳


In [5]:
wrong = events[2]['value']  # 模拟到达顺序策略让晚到上海事件直接覆盖当前地址。
correct = resolve(events[:2] + [events[2]])[0]['value']  # 读取时间 resolver 对同一子序列的正确结果。
print('失败案例：到达顺序=', wrong, '，时间 resolver=', correct)  # 展示晚到旧事件不会回滚新事实。
print('生产差距：需可信时钟、版本向量/ETag、来源签名、删除合规与冲突审计。')  # 说明完整更新系统。

失败案例：到达顺序= 上海 ，时间 resolver= 北京
生产差距：需可信时钟、版本向量/ETag、来源签名、删除合规与冲突审计。


In [6]:
assert current['value'] == '深圳'  # 验证最新权威更新成为最终状态。
assert history[3][1] == '北京'  # 验证模型推测广州不会覆盖权威北京。
assert current['source'] == 'authority'  # 验证最终事实来自权威来源。